# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR\(^2\) dataset using the `mlcroissant` library. All Croissant objects are referenced by their `@id` as required by the Croissant format for unambiguous referencing.

### Dataset Source
The dataset is described by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This also prints the overall dataset title and description referenced directly from the Croissant metadata.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review the available record sets and fields. Each is referenced by its unique `@id`. This section will list all record sets, then their corresponding fields and columns, to provide a map of the dataset structure.

In [ ]:
# List all record sets in the dataset using their @id
record_set_ids = []
print('Available Record Sets (by @id and name):')
for record_set in metadata.record_sets:
    rs_id = record_set['@id']
    rs_name = record_set.get('name', '')
    print(f"  @id: {rs_id}, name: {rs_name}")
    record_set_ids.append(rs_id)

print('\nFields and Columns for each record set:')
for record_set in metadata.record_sets:
    print(f"\nRecord Set @id: {record_set['@id']}, name: {record_set.get('name', '')}")
    if 'fields' in record_set:
        for field in record_set['fields']:
            f_id = field['@id']
            col_ids = [col['@id'] for col in field.get('columns', [])] if 'columns' in field else []
            print(f"  Field @id: {f_id}, name: {field.get('name', '')}, Columns: {col_ids}")
    else:
        print("  No fields found.")

## 3. Data Extraction
Load tabular data from one or more record sets into pandas DataFrames. Use the `@id` of each record set as required for all data referencing.

In [ ]:
# Prepare DataFrames from each record set by @id
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        # Defensive: only add DataFrame if records are present
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print("  No records found.")

# For demonstration, pick the first non-empty record set for further analysis
main_record_set_id = None
for k, v in dataframes.items():
    if len(v) > 0:
        main_record_set_id = k
        break

if main_record_set_id:
    print(f"\nMain record set for EDA: {main_record_set_id}")
    print(dataframes[main_record_set_id].head())
else:
    print("No tabular record set data found.")

## 4. Exploratory Data Analysis (EDA)
Perform common data processing such as filtering, normalization and simple grouping. All field/column references use their Croissant `@id` values. You can update the field choices below to analyze different fields as appropriate for the dataset.

In [ ]:
# Select a numeric field and a grouping field by their @id
import numpy as np

# Example: Use 'cr:field:coverage_percent' as numeric, 'cr:field:accession' as group if present
numeric_field_id = None
group_field_id = None

candidate_numeric_fields = [col for col in dataframes[main_record_set_id].columns if 'coverage' in col or 'MW' in col or 'Peptides' in col or 'Abundance' in col or 'Score' in col or 'count' in col]
if candidate_numeric_fields:
    numeric_field_id = candidate_numeric_fields[0]
    print(f"Auto-selected numeric field: {numeric_field_id}")

candidate_group_fields = [col for col in dataframes[main_record_set_id].columns if 'accession' in col or 'description' in col or 'sample' in col or 'group' in col]
if candidate_group_fields:
    group_field_id = candidate_group_fields[0]
    print(f"Auto-selected group-by field: {group_field_id}")

df = dataframes[main_record_set_id]

if numeric_field_id is not None:
    print(f"\n=== Filtering records with {numeric_field_id} > threshold ===")
    # Convert to numeric (errors='coerce')
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = np.nanpercentile(df[numeric_field_id], 50)  # median as example
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records ({numeric_field_id} > {threshold:.3f}): {len(filtered_df)} records")
    print(filtered_df[[numeric_field_id]].head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped (mean {numeric_field_id}) by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field found for analysis.")

## 5. Visualization
Visualize data distributions and relationships using the extracted DataFrame. Here we show a histogram of the main numeric field and, if possible, a barplot by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()
    
    if group_field_id is not None and group_field_id in df.columns:
        means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False).head(10)
        means.plot(kind='bar', figsize=(9, 4), title=f"Top 10 mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()

## 6. Conclusion
We loaded and explored the FAIR^2 protein mass spectrometry dataset using the `mlcroissant` library with full Croissant schema compatibility. Data extraction by record set and field `@id` enables precise access. The analysis demonstrates simple analytical patterns and visualizations, and you can further refine EDA by referencing additional field `@id`s noted in the overview above.
